# Digitdeck · Entregable M1 — Fine-tuning con LoRA

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · SI4006 · Semestre 2026-2

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

**Equipo:** Maximiliano Bustamante · Valeria Frances Hornung · Sebastián Castaño · Alejandro Posada

**Repositorio:** [AlejandroSPosada/Topicos_IA_Proyecto_Digitdeck](https://github.com/AlejandroSPosada/Topicos_IA_Proyecto_Digitdeck)

---

Este notebook documenta el trabajo del equipo para el Módulo 1 del curso: selección y
justificación del modelo base, construcción del dataset, fine-tuning con LoRA y evaluación
comparativa contra baselines.

> **Requisito:** activar GPU antes de ejecutar.
> `Runtime → Change runtime type → T4 GPU`
> Sin GPU el entrenamiento de las secciones 3 y 4 tardará mucho.

## 0 · Entorno de trabajo

In [1]:
# Instalación del ecosistema Hugging Face. ~1-2 min la primera vez.
# Si Colab pide RESTART RUNTIME, reinicia y vuelve a ejecutar desde esta celda.
!pip install -q transformers datasets peft accelerate evaluate bitsandbytes sentencepiece 2>/dev/null
print("Librerías instaladas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.9 MB/s eta 0:00:00
Librerías instaladas.


In [2]:
# Registro del entorno de entrenamiento para garantizar reproducibilidad.
import importlib.metadata as meta
import sys, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=== Entorno de entrenamiento ===")
print(f"Python      : {sys.version.split()[0]}")
print(f"Dispositivo : {DEVICE}", end="")
print(f" | {torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else " [AVISO: sin GPU activa]")
print()
for pkg in ["torch", "transformers", "peft", "datasets", "accelerate", "evaluate"]:
    try:
        print(f"  {pkg:<14} {meta.version(pkg)}")
    except meta.PackageNotFoundError:
        print(f"  {pkg:<14} FALTA")

=== Entorno de entrenamiento ===
Python      : 3.12.13
Dispositivo : cuda | Tesla T4

  torch          2.11.0+cu128
  transformers   5.13.1
  peft           0.19.1
  datasets       4.0.0
  accelerate     1.14.0
  evaluate       0.4.6


---

# 1 · Selección y justificación del modelo base

## 1.1 · Candidatos evaluados

Se evalúan cinco modelos que cubren distintas familias, tamaños y estrategias de cobertura
del español, todos dentro del presupuesto de la GPU T4 (≤16 GB VRAM):

| # | Modelo | Params | Familia | Cobertura español | Licencia |
|---|---|---|---|---|---|
| 1 | `Qwen/Qwen2.5-0.5B` | 0.5 B | Qwen2.5 | Multilingüe | Apache 2.0 |
| 2 | `meta-llama/Llama-3.2-1B` | 1 B | LLaMA 3.2 | Multilingüe | Llama 3.2 Community |
| 3 | `HuggingFaceTB/SmolLM2-1.7B` | 1.7 B | SmolLM2 | Multilingüe | Apache 2.0 |
| 4 | `BSC-LT/salamandra-2b` | 2 B | Salamandra | **Solo español/catalán** | Apache 2.0 |
| 5 | `microsoft/Phi-3.5-mini-instruct` | 3.8 B | Phi-3.5 | Multilingüe | MIT |

**Criterios de inclusión de la lista:**
- Rango de 0.5 B a 3.8 B, compatible con la GPU T4 de Colab gratuito.
- Al menos tres familias arquitectónicas distintas para que la comparación tenga poder discriminativo.
- Inclusión de `Salamandra`, el único modelo entrenado exclusivamente en español/catalán, como
  punto de referencia para evaluar si la especialización monolingüe compensa frente a
  los modelos multilingües.
- La decisión final se toma después del análisis de tokenizadores, no antes.

> **AVISO — `Llama-3.2-1B`:** requiere aceptar la licencia en Hugging Face antes de
> descargarlo. Visitar [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)
> y hacer clic en `Agree and access repository`. Luego autenticarse en Colab con
> `from huggingface_hub import login; login()`.

## 1.2 · Atributos registrados desde las model cards

In [3]:
# Datos extraídos de las model cards oficiales en Hugging Face.
CANDIDATOS = [
    {
        "id":       "Qwen/Qwen2.5-0.5B",
        "params_b": 0.49,
        "disco_gb": 1.0,
        "familia":  "Qwen2.5",
        "espanol":  "Multilingüe (>29 idiomas)",
        "licencia": "Apache 2.0",
        "hf_url":   "https://huggingface.co/Qwen/Qwen2.5-0.5B",
    },
    {
        "id":       "meta-llama/Llama-3.2-1B",
        "params_b": 1.24,
        "disco_gb": 2.5,
        "familia":  "LLaMA 3.2",
        "espanol":  "Multilingüe (8 idiomas incl. ES)",
        "licencia": "Llama 3.2 Community License",
        "hf_url":   "https://huggingface.co/meta-llama/Llama-3.2-1B",
    },
    {
        "id":       "HuggingFaceTB/SmolLM2-1.7B",
        "params_b": 1.71,
        "disco_gb": 3.4,
        "familia":  "SmolLM2",
        "espanol":  "Multilingüe (principalmente EN)",
        "licencia": "Apache 2.0",
        "hf_url":   "https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B",
    },
    {
        "id":       "BSC-LT/salamandra-2b",
        "params_b": 2.0,
        "disco_gb": 4.0,
        "familia":  "Salamandra (BSC)",
        "espanol":  "Español/Catalán — especializado",
        "licencia": "Apache 2.0",
        "hf_url":   "https://huggingface.co/BSC-LT/salamandra-2b",
    },
    {
        "id":       "microsoft/Phi-3.5-mini-instruct",
        "params_b": 3.82,
        "disco_gb": 7.6,
        "familia":  "Phi-3.5",
        "espanol":  "Multilingüe (>20 idiomas incl. ES)",
        "licencia": "MIT",
        "hf_url":   "https://huggingface.co/microsoft/Phi-3.5-mini-instruct",
    },
]

import pandas as pd
pd.set_option("display.max_colwidth", None)

df_candidatos = pd.DataFrame(CANDIDATOS).rename(columns={
    "id":       "Modelo",
    "params_b": "Params (B)",
    "disco_gb": "Disco (GB)",
    "familia":  "Familia",
    "espanol":  "Cobertura español",
    "licencia": "Licencia",
    "hf_url":   "Model card",
})
df_candidatos

,Modelo,Params (B),Disco (GB),Familia,Cobertura español,Licencia,Model card
0,Qwen/Qwen2.5-0.5B,0.49,1.0,Qwen2.5,Multilingüe (>29 idiomas),Apache 2.0,https://huggingface.co/Qwen/Qwen2.5-0.5B
1,meta-llama/Llama-3.2-1B,1.24,2.5,LLaMA 3.2,Multilingüe (8 idiomas incl. ES),Llama 3.2 Community License,https://huggingface.co/meta-llama/Llama-3.2-1B
2,HuggingFaceTB/SmolLM2-1.7B,1.71,3.4,SmolLM2,Multilingüe (principalmente EN),Apache 2.0,https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B
3,BSC-LT/salamandra-2b,2.00,4.0,Salamandra (BSC),Español/Catalán — especializado,Apache 2.0,https://huggingface.co/BSC-LT/salamandra-2b
4,microsoft/Phi-3.5-mini-instruct,3.82,7.6,Phi-3.5,Multilingüe (>20 idiomas incl. ES),MIT,https://huggingface.co/microsoft/Phi-3.5-mini-instruct


## 1.3 · Análisis de tokenizadores sobre texto de ecommerce en español

En clase se mostró que el tokenizador tiene un impacto directo en la calidad de las
representaciones y en el costo de inferencia: un vocabulario centrado en inglés fragmenta
palabras españolas en más piezas, lo que reduce la información semántica por token y
encarece el procesamiento.

Aplicamos ese análisis al dominio del proyecto: comparamos cómo cada tokenizador candidato
fragmenta texto real de ecommerce en español. Esta tabla constituye la evidencia empírica
de la decisión de selección de modelo.

In [4]:
from transformers import AutoTokenizer

# Solo se cargan los tokenizadores — sin pesos del modelo.
# Primera ejecución: ~1-2 min por descarga de archivos de vocabulario.
print("Cargando tokenizadores...")
tokenizadores = {}
for c in CANDIDATOS:
    try:
        tokenizadores[c["id"]] = AutoTokenizer.from_pretrained(c["id"])
        print(f"  OK  {c['id']:<45} vocab: {tokenizadores[c['id']].vocab_size:>9,} tokens")
    except Exception as e:
        print(f"  ERR {c['id']:<45} {e}")
        print(f"      -> Si es Llama, acepta la licencia en HF y autenticate con login().")

print(f"\n{len(tokenizadores)}/{len(CANDIDATOS)} tokenizadores cargados.")

Cargando tokenizadores...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

  OK  microsoft/Phi-3.5-mini-instruct               vocab:    32,000 tokens

4/5 tokenizadores cargados.


In [5]:
def comparar(texto, nombre_corto=True):
    """Muestra cómo cada tokenizador fragmenta el mismo texto."""
    filas = []
    for model_id, tok in tokenizadores.items():
        nombre = model_id.split("/")[-1] if nombre_corto else model_id
        piezas = tok.tokenize(texto)
        filas.append({
            "Modelo":   nombre,
            "N tokens": len(piezas),
            "Tokens":   " | ".join(piezas),
        })
    return pd.DataFrame(filas).set_index("Modelo")

In [6]:
# Muestra de 15 consultas representativas del dominio.
MUESTRA_DOMINIO = [
    "televisión 4K 55 pulgadas",
    "auriculares inalámbricos cancelación de ruido",
    "Samsung Galaxy S24 Ultra",
    "Xiaomi Redmi Note 13",
    "PlayStation 5 edición digital",
    "ASUS TUF Gaming F15 FX506",
    "HDMI 2.1 cable 2m",
    "smartwatch deportivo waterproof",
    "laptop gamer RTX 4060",
    "celular libre 5G doble SIM",
    "impresora multifuncional wifi tinta",
    "audifono bluetooth sony",
    "tableta grafica wacom",
    "procesador Intel Core i7 13ava generación socket 1700",
    "silla ergonómica gamer con soporte lumbar y reposapiés ajustable",
]

print(f"{len(MUESTRA_DOMINIO)} consultas preparadas.")

15 consultas preparadas.


In [7]:
# Fragmentación caso a caso — quitar el slice [:5] para ver todas las consultas.
for texto in MUESTRA_DOMINIO[:5]:
    print(f"\n--- '{texto}' ---")
    display(comparar(texto))


--- 'televisión 4K 55 pulgadas' ---


,N tokens,Tokens
Modelo,,
Qwen2.5-0.5B,11,tele | visiÃ³n | Ġ | 4 | K | Ġ | 5 | 5 | Ġpul | g | adas
SmolLM2-1.7B,13,tele | vis | i | Ã³n | Ġ | 4 | K | Ġ | 5 | 5 | Ġpul | g | adas
salamandra-2b,9,▁televisión | ▁ | 4 | K | ▁ | 5 | 5 | ▁pul | gadas
Phi-3.5-mini-instruct,11,▁televis | ión | ▁ | 4 | K | ▁ | 5 | 5 | ▁pul | g | adas



--- 'auriculares inalámbricos cancelación de ruido' ---


,N tokens,Tokens
Modelo,,
Qwen2.5-0.5B,13,aur | ic | ulares | Ġin | al | Ã¡m | br | icos | Ġcancel | aciÃ³n | Ġde | Ġr | uido
SmolLM2-1.7B,15,aur | ic | ula | res | Ġin | al | Ã¡ | mb | ric | os | Ġcancel | aciÃ³n | Ġde | Ġru | ido
salamandra-2b,9,▁au | riculares | ▁inal | ám | b | ricos | ▁cancelación | ▁de | ▁ruido
Phi-3.5-mini-instruct,13,▁aur | icular | es | ▁in | al | ám | br | icos | ▁cancel | ación | ▁de | ▁ru | ido



--- 'Samsung Galaxy S24 Ultra' ---


,N tokens,Tokens
Modelo,,
Qwen2.5-0.5B,6,Samsung | ĠGalaxy | ĠS | 2 | 4 | ĠUltra
SmolLM2-1.7B,8,S | ams | ung | ĠGalaxy | ĠS | 2 | 4 | ĠUltra
salamandra-2b,6,▁Samsung | ▁Galaxy | ▁S | 2 | 4 | ▁Ultra
Phi-3.5-mini-instruct,9,▁S | amsung | ▁Gal | axy | ▁S | 2 | 4 | ▁Ult | ra



--- 'Xiaomi Redmi Note 13' ---


,N tokens,Tokens
Modelo,,
Qwen2.5-0.5B,9,X | ia | omi | ĠRed | mi | ĠNote | Ġ | 1 | 3
SmolLM2-1.7B,9,X | ia | omi | ĠRed | mi | ĠNote | Ġ | 1 | 3
salamandra-2b,6,▁Xiaomi | ▁Redmi | ▁Note | ▁ | 1 | 3
Phi-3.5-mini-instruct,9,▁X | ia | omi | ▁Red | mi | ▁Note | ▁ | 1 | 3



--- 'PlayStation 5 edición digital' ---


,N tokens,Tokens
Modelo,,
Qwen2.5-0.5B,6,Play | Station | Ġ | 5 | ĠediciÃ³n | Ġdigital
SmolLM2-1.7B,8,Play | Station | Ġ | 5 | Ġed | ici | Ã³n | Ġdigital
salamandra-2b,5,▁PlayStation | ▁ | 5 | ▁edición | ▁digital
Phi-3.5-mini-instruct,6,▁Play | Station | ▁ | 5 | ▁edición | ▁digital


In [8]:
# Costo total sobre la muestra completa: menor es mejor.
filas_total = []
for model_id, tok in tokenizadores.items():
    conteos  = [len(tok.tokenize(t)) for t in MUESTRA_DOMINIO]
    total    = sum(conteos)
    promedio = total / len(conteos)
    nombre   = model_id.split("/")[-1]
    filas_total.append({
        "Modelo":         nombre,
        "Total tokens":   total,
        "Promedio/frase": round(promedio, 1),
        "Vocab size":     f"{tok.vocab_size:,}",
    })

df_total = pd.DataFrame(filas_total).set_index("Modelo").sort_values("Total tokens")
df_total.style.background_gradient(subset=["Total tokens", "Promedio/frase"], cmap="YlOrRd")

,Total tokens,Promedio/frase,Vocab size
Modelo,,,
salamandra-2b,126,8.400000,"256,000"
Qwen2.5-0.5B,149,9.900000,"151,643"
Phi-3.5-mini-instruct,161,10.700000,"32,000"
SmolLM2-1.7B,170,11.300000,"49,152"


In [9]:
# Costo por término crítico del dominio.
TERMINOS_CRITICOS = [
    "televisión", "auriculares", "inalámbricos", "ergonómica",
    "Samsung", "Xiaomi", "PlayStation", "RTX",
    "smartwatch", "laptop", "bluetooth", "waterproof",
]

filas_criticos = []
for term in TERMINOS_CRITICOS:
    fila = {"Término": term}
    for model_id, tok in tokenizadores.items():
        fila[model_id.split("/")[-1]] = len(tok.tokenize(term))
    filas_criticos.append(fila)

df_criticos = pd.DataFrame(filas_criticos).set_index("Término")
print("Número de tokens por término — menor indica mejor cobertura del dominio:")
df_criticos.style.background_gradient(cmap="YlOrRd", axis=1)

Número de tokens por término — menor indica mejor cobertura del dominio:


,Qwen2.5-0.5B,SmolLM2-1.7B,salamandra-2b,Phi-3.5-mini-instruct
Término,,,,
televisión,2,4,1,2
auriculares,3,4,2,3
inalámbricos,4,5,4,5
ergonómica,4,5,2,4
Samsung,1,3,1,2
Xiaomi,3,3,1,3
PlayStation,2,2,1,2
RTX,2,2,1,2
smartwatch,2,2,2,2


## 1.4 · Justificación del modelo elegido

### Hallazgos del análisis de tokenizadores

Se cargaron 4 de los 5 tokenizadores (`Llama-3.2-1B` requiere autenticación con token HF por
ser un modelo con acceso restringido). Los resultados sobre las 15 consultas de dominio son:

**Tabla de costo total (15 consultas de ecommerce en español):**

| Modelo | Total tokens | Promedio/frase | Vocab size |
|---|---|---|---|
| `salamandra-2b` | **126** | **8.4** | 256,000 |
| `Qwen2.5-0.5B` | 149 | 9.9 | 151,643 |
| `Phi-3.5-mini-instruct` | 161 | 10.7 | 32,000 |
| `SmolLM2-1.7B` | 170 | 11.3 | 49,152 |

**Hallazgos clave:**

- **Salamandra-2b tokeniza mejor el español en todos los frentes:** palabras con tilde (`televisión` → 1 token, vs 2 en Qwen y Phi), marcas (`Xiaomi` → 1 token, vs 3 en Qwen), y anglicismos de ecommerce (`laptop` → 1 token). Su vocabulario de 256,000 entradas refleja cobertura extensiva del dominio hispanohablante.
- **SmolLM2-1.7B es el peor tokenizador para español:** fragmenta más que cualquier otro en tildes (`inalámbricos` → 5 tokens), marcas (`Samsung` → 3 tokens) y consultas largas. Su entrenamiento es principalmente en inglés.
- **Qwen2.5-0.5B ocupa el segundo lugar**, con un costo 18% superior al de Salamandra pero con sólo 0.49 B de parámetros.
- **Phi-3.5-mini-instruct** tiene el vocabulario más pequeño (32,000 tokens), lo que lo hace fragmentar más que Salamandra o Qwen a pesar de ser el modelo más grande.

### Preguntas que el argumento debe resistir

**1. ¿Por qué no elegir `Salamandra-2B` si claramente es el mejor tokenizador para español?**

Salamandra es el candidato más fuerte en tokenización, pero fue entrenado exclusivamente en español
y catalán. Digitdeck opera en un dominio donde los términos técnicos, marcas internacionales y
anglicismos son frecuentes (RTX, HDMI, smartwatch, gaming). Un modelo sin exposición al inglés
puede tener representaciones internas pobres para estos términos, incluso si su tokenizador los
encapsula bien. Adicionalmente, con 2B parámetros y un vocabulario de 256K, su tiempo de
fine-tuning en la T4 de Colab gratuito será el más alto del grupo.

**2. ¿Por qué no elegir `Phi-3.5-mini-instruct` si es el más grande (3.8 B)?**

Phi-3.5-mini tiene el vocabulario más pequeño (32,000 tokens), lo que lo hace el segundo peor
tokenizador del grupo para español. Con 3.8 B de parámetros y 7.6 GB en disco, es el más costoso
de fine-tunear: en una T4 con 16 GB de VRAM, cargar el modelo base más los adaptadores LoRA y
las activaciones del batch deja poco margen. El beneficio de sus parámetros adicionales no
justifica el costo operativo para una primera iteración de M1.

### Decisión

El equipo elige **`Qwen/Qwen2.5-0.5B`** como modelo base por las siguientes razones:

1. **Costo de cómputo mínimo:** 0.49 B de parámetros y 1 GB en disco — el más liviano del grupo.
2. **Tokenización competente para el dominio:** segundo mejor resultado en costo total (149 tokens
   vs 126 de Salamandra), con una diferencia del 18% que se compensa con su velocidad de entrenamiento.
3. **Multilingüe con cobertura real del español:** vocabulario de 151,643 tokens, con exposición al
   inglés necesaria para manejar la terminología técnica del ecommerce.
4. **Licencia Apache 2.0:** sin restricciones de uso comercial ni requisito de autenticación.
5. **Trazabilidad con el material de clase:** el pipeline de fine-tuning visto en clase usa
   exactamente este modelo como referencia.

### Limitaciones conocidas

- El análisis no incluyó `Llama-3.2-1B` por la barrera de acceso (modelo gated). Queda como trabajo futuro.
- Con 0.49 B de parámetros, puede tener capacidad de razonamiento limitada para pares consulta-producto
  muy ambiguos. Si los resultados son insuficientes, el siguiente candidato es `Qwen/Qwen2.5-1.5B`.
- La muestra de 15 consultas es representativa pero pequeña.

---

# 2 · Dataset: construcción y documentación

## 2.0 · Descripción del dataset

Se usa el **Amazon Shopping Queries Dataset (ESCI)**, publicado por Amazon Science bajo
licencia Apache 2.0. Es el único dataset público de relevancia de búsqueda con anotaciones
en español a escala suficiente para fine-tuning.

| Atributo | Detalle |
|---|---|
| **Fuente** | [amazon-science/esci-data](https://github.com/amazon-science/esci-data) |
| **Licencia** | Apache 2.0 |
| **Idiomas** | Inglés, Español, Japonés |
| **Subconjunto ES** | ~8,049 queries · ~218,774 pares query-producto |
| **Etiquetas** | `E` Exact · `S` Substitute · `C` Complement · `I` Irrelevant |
| **Campos usados** | `query`, `product_title`, `product_locale`, `esci_label`, `split` |
| **Splits** | `train` y `test` predefinidos en el dataset |

**Mapeo de etiquetas ESCI a formato binario para fine-tuning causal:**

| ESCI | Significado | Etiqueta modelo |
|---|---|---|
| `E` (Exact) | Producto que responde directamente la consulta | `relevante` |
| `S` (Substitute) | Producto alternativo, posiblemente útil | `no relevante` |
| `C` (Complement) | Producto complementario | `no relevante` |
| `I` (Irrelevant) | Sin relación con la consulta | `no relevante` |

Se usa el mapeo binario `E → relevante`, `S+C+I → no relevante` para simplificar la tarea
en M1. La pérdida de información en `S` y `C` se documenta como limitación conocida.

## 2.1 · Descarga del dataset ESCI

Los archivos están en Git LFS. Se descargan directamente con `wget` usando la URL de medios
de GitHub (`media.githubusercontent.com`). Tiempos aproximados en Colab T4:
- `examples.parquet` (~300 MB): ~1 min
- `products.parquet` (~1.7 GB): ~5-8 min

In [14]:
import os

BASE_URL = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset"
ARCHIVOS = [
    "shopping_queries_dataset_examples.parquet",
    "shopping_queries_dataset_products.parquet",
]

for archivo in ARCHIVOS:
    # Si existe pero es corrupto (menos de 1 MB), se borra para forzar re-descarga
    if os.path.exists(archivo) and os.path.getsize(archivo) < 1e6:
        os.remove(archivo)

    if os.path.exists(archivo):
        mb = os.path.getsize(archivo) / 1e6
        print(f"  Ya existe: {archivo} ({mb:.1f} MB) — omitiendo descarga.")
    else:
        print(f"  Descargando {archivo} ...")
        # -L es CRUCIAL para que wget/curl siga las redirecciones de GitHub LFS
        !wget -q -L --show-progress -O "{archivo}" "{BASE_URL}/{archivo}"

        if os.path.exists(archivo):
            mb = os.path.getsize(archivo) / 1e6
            print(f"  Descargado: {archivo} ({mb:.1f} MB)\n")

print("Descarga completada con éxito.")

  Descargando shopping_queries_dataset_examples.parquet ...
shopping_queries_da 100%[===================>]  48.91M  32.5MB/s    in 1.5s    
  Descargado: shopping_queries_dataset_examples.parquet (51.3 MB)

  Descargando shopping_queries_dataset_products.parquet ...
shopping_queries_da 100%[===================>]   1.03G  52.9MB/s    in 19s     
  Descargado: shopping_queries_dataset_products.parquet (1108.9 MB)

Descarga completada con éxito.


## 2.2 · Carga y filtrado del subconjunto español

In [15]:
import pandas as pd

# --- Cargar ejemplos (pares query-producto con etiqueta) ---
print("Cargando examples.parquet ...")
df_ex = pd.read_parquet(
    "shopping_queries_dataset_examples.parquet",
    columns=["query_id", "product_id", "query", "esci_label", "split", "product_locale"],
)
print(f"  Total pares (todos los idiomas): {len(df_ex):,}")

# Filtrar subconjunto español
df_es = df_ex[df_ex["product_locale"] == "es"].copy()
print(f"  Pares en español (es):           {len(df_es):,}")
print(f"  Queries únicas (es):             {df_es['query_id'].nunique():,}")
print(f"  Distribución de splits:")
print(df_es["split"].value_counts().to_string())
print(f"  Distribución de etiquetas ESCI:")
print(df_es["esci_label"].value_counts().to_string())

Cargando examples.parquet ...
  Total pares (todos los idiomas): 2,621,288
  Pares en español (es):           356,410
  Queries únicas (es):             15,180
  Distribución de splits:
split
train    263063
test      93347
  Distribución de etiquetas ESCI:
esci_label
E    202066
S     89059
I     44782
C     20503


In [16]:
# --- Cargar productos (títulos y descripción) ---
# Solo cargamos product_id y product_title para minimizar uso de memoria.
print("Cargando products.parquet (solo columnas necesarias) ...")
product_ids_es = set(df_es["product_id"].unique())

df_prod = pd.read_parquet(
    "shopping_queries_dataset_products.parquet",
    columns=["product_id", "product_title", "product_locale"],
    filters=[("product_locale", "==", "es")],
)
print(f"  Productos en español cargados: {len(df_prod):,}")

# Join: ejemplos + títulos de producto
df = df_es.merge(df_prod[["product_id", "product_title"]], on="product_id", how="left")
print(f"  Pares con título de producto:  {len(df):,}")
print(f"  Pares sin título (nulos):      {df['product_title'].isna().sum():,}")

Cargando products.parquet (solo columnas necesarias) ...
  Productos en español cargados: 260,011
  Pares con título de producto:  356,410
  Pares sin título (nulos):      0


## 2.3 · Criterios de inclusión/exclusión y limpieza reproducible

In [17]:
import re

SEMILLA = 42  # semilla fija para cualquier submuestreo adicional

def limpiar_texto(texto: str) -> str:
    """Normalización reproducible: minúsculas, colapso de espacios."""
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def n_palabras(texto: str) -> int:
    """Cuenta palabras en un texto ya limpio."""
    return len(texto.split()) if isinstance(texto, str) else 0


n_antes = len(df)

# 1. Eliminar filas con product_title nulo
df = df.dropna(subset=["product_title"])
print(f"Tras eliminar titles nulos:         {len(df):,}  (-{n_antes - len(df):,})")

# 2. Normalizar texto
df["query"]         = df["query"].map(limpiar_texto)
df["product_title"] = df["product_title"].map(limpiar_texto)

# 3. Descartar product_title con menos de 3 palabras
n_antes = len(df)
df = df[df["product_title"].map(n_palabras) >= 3]
print(f"Tras filtrar titles cortos (<3 palabras): {len(df):,}  (-{n_antes - len(df):,})")

# 4. Deduplicar por (query_id, product_id)
n_antes = len(df)
df = df.drop_duplicates(subset=["query_id", "product_id"])
print(f"Tras deduplicar (query_id, product_id): {len(df):,}  (-{n_antes - len(df):,})")

print(f"\nDataset limpio final: {len(df):,} pares")
print(df["split"].value_counts().to_string())

Tras eliminar titles nulos:         356,410  (-0)
Tras filtrar titles cortos (<3 palabras): 354,288  (-2,122)
Tras deduplicar (query_id, product_id): 354,288  (-0)

Dataset limpio final: 354,288 pares
split
train    261549
test      92739


## 2.4 · Mapeo de etiquetas y formateo para fine-tuning causal

In [18]:
# Mapeo ESCI → binario
MAPEO_ETIQUETA = {
    "E": "relevante",      # Exact match
    "S": "no relevante",   # Substitute
    "C": "no relevante",   # Complement
    "I": "no relevante",   # Irrelevant
}

df["etiqueta"] = df["esci_label"].map(MAPEO_ETIQUETA)

print("Distribución de etiquetas tras el mapeo binario:")
print(df["etiqueta"].value_counts().to_string())
pct = df["etiqueta"].value_counts(normalize=True).mul(100).round(1)
print(f"\nBalance: {pct.get('relevante', 0):.1f}% relevante / {pct.get('no relevante', 0):.1f}% no relevante")

Distribución de etiquetas tras el mapeo binario:
etiqueta
relevante       200936
no relevante    153352

Balance: 56.7% relevante / 43.3% no relevante


In [19]:
# Plantilla de texto para fine-tuning causal.
# El modelo aprende a predecir la etiqueta como continuación del texto.
PLANTILLA = "Consulta: {query}\nProducto: {product_title}\nRelevancia: {etiqueta}"

def formatear(fila):
    return PLANTILLA.format(
        query        = fila["query"],
        product_title= fila["product_title"],
        etiqueta     = fila["etiqueta"],
    )

df["texto"] = df.apply(formatear, axis=1)

# Mostrar ejemplos representativos
print("--- Ejemplo: RELEVANTE ---")
print(df[df["etiqueta"] == "relevante"]["texto"].iloc[0])
print("\n--- Ejemplo: NO RELEVANTE ---")
print(df[df["etiqueta"] == "no relevante"]["texto"].iloc[0])
print(f"\nLongitud promedio del texto formateado: {df['texto'].str.len().mean():.0f} chars")

--- Ejemplo: RELEVANTE ---
Consulta: !solid camiseta sin manga
Producto: tuc tuc punto tropicool camiseta sin mangas para bebés y niños pequeños, blanco, 4a
Relevancia: relevante

--- Ejemplo: NO RELEVANTE ---
Consulta: !solid camiseta sin manga
Producto: aloha hawaii piña con gafas de sol regalo de playa #3 camiseta sin mangas
Relevancia: no relevante

Longitud promedio del texto formateado: 186 chars


## 2.5 · Generación de splits con semilla fija

In [20]:
# El dataset ya viene con splits predefinidos (train/test).
# Los usamos directamente para garantizar que la evaluación sea comparable
# con otros trabajos que usen el mismo dataset.

df_train = df[df["split"] == "train"][["query_id", "product_id", "query", "product_title", "etiqueta", "texto"]].reset_index(drop=True)
df_test  = df[df["split"] == "test" ][["query_id", "product_id", "query", "product_title", "etiqueta", "texto"]].reset_index(drop=True)

print(f"Split train : {len(df_train):,} pares")
print(f"  relevante    : {(df_train['etiqueta']=='relevante').sum():,} ({(df_train['etiqueta']=='relevante').mean()*100:.1f}%)")
print(f"  no relevante : {(df_train['etiqueta']=='no relevante').sum():,} ({(df_train['etiqueta']=='no relevante').mean()*100:.1f}%)")
print()
print(f"Split test  : {len(df_test):,} pares")
print(f"  relevante    : {(df_test['etiqueta']=='relevante').sum():,} ({(df_test['etiqueta']=='relevante').mean()*100:.1f}%)")
print(f"  no relevante : {(df_test['etiqueta']=='no relevante').sum():,} ({(df_test['etiqueta']=='no relevante').mean()*100:.1f}%)")
print(f"\nSemilla para submuestreo adicional: SEMILLA = {SEMILLA}")

Split train : 261,549 pares
  relevante    : 151,596 (58.0%)
  no relevante : 109,953 (42.0%)

Split test  : 92,739 pares
  relevante    : 49,340 (53.2%)
  no relevante : 43,399 (46.8%)

Semilla para submuestreo adicional: SEMILLA = 42


In [21]:
# Submuestreo opcional para experimentos rápidos en Colab (descomenta si necesitas).
# Usa SEMILLA para garantizar reproducibilidad.
#
# N_TRAIN = 10_000
# N_TEST  =  2_000
# df_train = df_train.sample(n=N_TRAIN, random_state=SEMILLA).reset_index(drop=True)
# df_test  = df_test.sample(n=N_TEST,  random_state=SEMILLA).reset_index(drop=True)
# print(f"Submuestreado: {len(df_train):,} train / {len(df_test):,} test  (seed={SEMILLA})")

print("Submuestreo desactivado — usando el dataset completo.")

Submuestreo desactivado — usando el dataset completo.


## 2.6 · Verificación de reproducibilidad

In [22]:
import hashlib

def hash_df(df: pd.DataFrame) -> str:
    """Hash MD5 del contenido del DataFrame para verificar reproducibilidad."""
    contenido = df.to_csv(index=False).encode()
    return hashlib.md5(contenido).hexdigest()

hash_train = hash_df(df_train)
hash_test  = hash_df(df_test)

print("=== Verificación de reproducibilidad ===")
print(f"Hash MD5 df_train : {hash_train}")
print(f"Hash MD5 df_test  : {hash_test}")
print()
print("Para confirmar reproducibilidad, correr este notebook desde cero y verificar")
print("que los hashes anteriores coincidan con los de esta ejecución.")

=== Verificación de reproducibilidad ===
Hash MD5 df_train : ea81544ebffd9800dc201d2d736c7703
Hash MD5 df_test  : d3e6c6d42fb62b90127cff842381460e

Para confirmar reproducibilidad, correr este notebook desde cero y verificar
que los hashes anteriores coincidan con los de esta ejecución.


## 2.7 · Guardado de la muestra y documentación

In [23]:
import os

# Guardar una muestra pequeña versionable (los parquet completos no se versiona por tamaño).
# La muestra sirve como smoke-test y como ejemplo del formato para el repositorio.
os.makedirs("Datos", exist_ok=True)

N_MUESTRA = 200   # 100 train + 100 test, balanceados
muestra_train = (
    df_train
    .groupby("etiqueta", group_keys=False)
    .apply(lambda g: g.sample(min(N_MUESTRA // 2, len(g)), random_state=SEMILLA))
    .reset_index(drop=True)
)
muestra_test = (
    df_test
    .groupby("etiqueta", group_keys=False)
    .apply(lambda g: g.sample(min(N_MUESTRA // 2, len(g)), random_state=SEMILLA))
    .reset_index(drop=True)
)

muestra_train.to_csv("Datos/esci_es_muestra_train.csv", index=False)
muestra_test.to_csv( "Datos/esci_es_muestra_test.csv",  index=False)

print(f"Muestra train guardada: {len(muestra_train)} pares -> Datos/esci_es_muestra_train.csv")
print(f"Muestra test  guardada: {len(muestra_test)} pares -> Datos/esci_es_muestra_test.csv")
print()
print("NOTA: los archivos .parquet completos NO se versionan (>1 GB).")
print("      Descargalos de https://github.com/amazon-science/esci-data")

Muestra train guardada: 200 pares -> Datos/esci_es_muestra_train.csv
Muestra test  guardada: 200 pares -> Datos/esci_es_muestra_test.csv

NOTA: los archivos .parquet completos NO se versionan (>1 GB).
      Descargalos de https://github.com/amazon-science/esci-data


/tmp/ipykernel_2865/3415259929.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(N_MUESTRA // 2, len(g)), random_state=SEMILLA))
/tmp/ipykernel_2865/3415259929.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(N_MUESTRA // 2, len(g)), random_state=SEMILLA))


## 2.8 · Limitaciones conocidas del dataset

Las siguientes limitaciones se documentan explícitamente antes de usar el dataset en
el entrenamiento. Ignorarlas sería un error de diseño que afectaría la interpretación
de los resultados.

**1. Origen del dataset — Amazon US/ES**
El dataset proviene de búsquedas reales en Amazon, que opera principalmente en mercados
de España y Estados Unidos hispanohablantes. El catálogo de Digitdeck puede tener
distribuciones diferentes (categorías distintas, marcas propias, terminología local
colombiana). El modelo puede rendir peor en categorías subrepresentadas en el dataset ESCI.

**2. Mapeo binario pierde información**
El mapeo `E → relevante`, `S+C+I → no relevante` trata igual un producto sustituto (útil)
y uno completamente irrelevante. Esto puede hacer que el modelo aprenda señales más ruidosas
para casos ambiguos (`S` y `C`). La versión de 4 clases queda como mejora futura.

**3. Desbalance de clases**
El dataset no está balanceado: hay más pares no relevantes que relevantes. El entrenamiento
sin corrección tenderá a predecir `no relevante` más frecuentemente. Se puede mitigar con
pesos de clase o submuestreo balanceado.

**4. Texto de producto truncado**
Solo se usa `product_title`. La descripción completa (`product_description`) contiene
información relevante pero encarece el entrenamiento. Para M1 se prioriza la eficiencia
sobre la completitud.